# Ch.4 機械学習モデル構築（scikit-learn）

## 本チャプターのゴール
- 機械学習のワークフロー（前処理→学習→予測→評価）を一通り体験する
- モデルの評価結果を読み解き、改善の仮説を立てられる
- 「モデルを作って終わり」ではなく、結果の解釈まで行う

## 使用データ
引き続きワインデータセット（`load_wine`）
- **課題**：化学成分データからワインの品種（3クラス）を予測する
- **アルゴリズム**：ランダムフォレスト（`RandomForestClassifier`）

## ワークフロー全体像
```
データ準備 → 前処理（スケーリング）→ データ分割 → モデル学習
         → 予測 → 評価（Accuracy・混同行列）→ 特徴量重要度の確認
```

---

## 🤖 生成AIの使い方ガイド（本チャプター）

| AIに任せてよいこと | 自分で理解すること |
|---|---|
| `StandardScaler` のコードの書き方 | なぜスケーリングが必要か（数値の大きさの差） |
| `confusion_matrix` の表示方法 | 混同行列の読み方（どこが誤分類か） |
| `feature_importances_` の取得と可視化 | どの特徴量が重要で、それはなぜか |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# データ準備
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

X = df[wine.feature_names]  # 特徴量（13列）
y = df['target']             # ターゲット（品種: 0/1/2）

print('特徴量の形状:', X.shape)
print('ターゲットの分布:')
print(y.value_counts().sort_index())

---
## 4.1 データ分割（Train / Test）

モデルの「本当の性能」を測るために、学習に使うデータと評価に使うデータを分けます。

In [ ]:
# 訓練データ 80% / テストデータ 20% に分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% をテストデータに
    random_state=42,  # 再現性のための乱数シード
    stratify=y        # 品種の割合を均等に保つ
)

print('訓練データ:', X_train.shape)
print('テストデータ:', X_test.shape)
print()
print('訓練データの品種分布:')
print(y_train.value_counts().sort_index())

---
## 4.2 前処理：標準化（StandardScaler）

**なぜ必要か？**
アルコール度数（約12〜15）とプロリン含有量（約278〜1680）では数値のスケールが全く違います。
スケーリングせずにモデルを学習すると、大きな値の変数が不当に影響力を持ちます。

In [ ]:
# スケールの差を確認
print('スケーリング前の値の範囲：')
print(X_train[['alcohol', 'proline']].describe().loc[['min', 'max']].round(1))

In [ ]:
# StandardScaler：平均0、標準偏差1 になるように変換
scaler = StandardScaler()

# 注意：fit は訓練データのみで行う（テストデータの情報は使わない）
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)      # transform のみ（fit はしない）

# スケーリング後の確認
print('スケーリング後の値の範囲（訓練データ）：')
X_train_df = pd.DataFrame(X_train_scaled, columns=wine.feature_names)
print(X_train_df[['alcohol', 'proline']].describe().loc[['mean', 'std', 'min', 'max']].round(2))

---
## 4.3 モデル学習（ランダムフォレスト）

**ランダムフォレストとは？**
- 多数の「決定木」を組み合わせたアンサンブル学習
- 各決定木がバラバラな予測をして、多数決で最終結果を決める
- 単純な決定木より精度が高く、過学習しにくい

In [ ]:
# モデルの定義と学習
model = RandomForestClassifier(
    n_estimators=100,   # 決定木の本数
    random_state=42
)

model.fit(X_train_scaled, y_train)
print('学習完了！')

---
## 4.4 評価：モデルの性能を測る

In [ ]:
# テストデータで予測
y_pred = model.predict(X_test_scaled)

# 正解率（Accuracy）
acc = accuracy_score(y_test, y_pred)
print(f'正解率（Accuracy）: {acc:.1%}')

In [ ]:
# 混同行列（Confusion Matrix）
cm = confusion_matrix(y_test, y_pred)

class_names = ['Barolo(0)', 'Grignolino(1)', 'Barbera(2)']

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm,
            annot=True, fmt='d',
            cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
ax.set_title('混同行列（Confusion Matrix）')
ax.set_xlabel('予測ラベル')
ax.set_ylabel('正解ラベル')
plt.tight_layout()
plt.show()

print('読み方：対角成分（左上〜右下）= 正解した件数')
print('       対角以外 = 誤分類した件数')

In [ ]:
# 詳細な評価レポート
print('=== 評価レポート ===')
print(classification_report(y_test, y_pred,
                             target_names=class_names))
print('precision: 予測が正解だった割合（False Positive を減らしたい）')
print('recall   : 実際の正解を拾えた割合（False Negative を減らしたい）')

---
## 4.5 特徴量重要度：何が予測に貢献しているか

In [ ]:
# 特徴量重要度の取得と可視化
importances = pd.Series(model.feature_importances_, index=wine.feature_names)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('特徴量重要度（Feature Importances）')
ax.set_xlabel('重要度')
plt.tight_layout()
plt.show()

print('最も重要な特徴量:', importances.idxmax())
print('\nCh.2のEDAで見たことと一致していますか？')

---
## 🖊️ 演習（40分）

---

### 問1：ハイパーパラメータを変えて精度の変化を確認する

In [ ]:
# TODO: n_estimators を [10, 50, 100, 200] で変えて、
#       それぞれの正解率を計算し、折れ線グラフで表示してください
# ヒント：for ループで n_estimators を変えながらモデルを学習・評価します

n_estimators_list = [10, 50, 100, 200]
accuracies = []

# TODO: ここにコードを書く



### 問2：スケーリングなしの場合と比較する

In [ ]:
# TODO: StandardScaler を使わずに X_train, X_test を直接使って
#       モデルを学習・評価し、スケーリングありの場合と正解率を比較してください
#
# スケーリングあり:  Accuracy = （先ほどの結果）
# スケーリングなし: Accuracy = ?



### 問3：混同行列を読み解く

元の混同行列を見て、以下に答えてください。

**Q1. どの品種が最も誤分類されやすいですか？**

→ 答え：

**Q2. 品種0（Barolo）が品種1（Grignolino）に誤分類されるケースがあるとしたら、実務ではどんな問題が起きますか？（自由記述）**

→ 答え：

**Q3. 正解率が高くても、モデルが「使えない」ケースはどんな時ですか？（例：不均衡データ）**

→ 答え：


### 問4（発展）：重要度が高い上位3特徴量だけでモデルを作り直す

In [ ]:
# TODO: feature_importances_ を使って重要度が高い上位3特徴量を特定し、
#       その3列だけを使ってモデルを学習・評価してください
#
# 上位3特徴量だけでも同等の精度が出るか確認しましょう



---
## ✅ チャプターのまとめ

| ステップ | コード |
|---------|--------|
| データ分割 | `train_test_split(X, y, test_size=0.2, stratify=y)` |
| 標準化（fit） | `scaler.fit_transform(X_train)` |
| 標準化（transform） | `scaler.transform(X_test)` |
| モデル定義 | `RandomForestClassifier(n_estimators=100)` |
| 学習 | `model.fit(X_train_scaled, y_train)` |
| 予測 | `model.predict(X_test_scaled)` |
| 正解率 | `accuracy_score(y_test, y_pred)` |
| 混同行列 | `confusion_matrix(y_test, y_pred)` |
| 特徴量重要度 | `model.feature_importances_` |

**重要なポイント**
- `scaler.fit` は**訓練データのみ**で行う
- テストデータは「未知のデータ」として扱う
- 正解率だけでなく、混同行列で誤分類のパターンを確認する
- 特徴量重要度でEDAの仮説を検証できる